first we need to compute global scaling of all: 10...105 sets

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# Path to your directory
# ------------------------------------------------------------------
base_path = Path("/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/mut_project_updates/figures/tooth model/plot_b/files_for_global_scaling")

# Get all OPC files
files = sorted(base_path.glob("*OPCandCusp.txt"))

# ------------------------------------------------------------------
# 1. Load the global reference dataset (105)
# ------------------------------------------------------------------
global_file = base_path / "105OPCandCusp.txt"

df_global = pd.read_csv(
    global_file,
    delim_whitespace=True,
    skiprows=1,  # skip header
    names=["ID", "Cusps", "OPC"]
)

global_min = df_global["OPC"].min()
global_max = df_global["OPC"].max()
global_N   = df_global["Cusps"].nunique()

print("Global scaling constants (from 105 dataset)")
print("------------------------------------------------")
print(f"global_min = {global_min}")
print(f"global_max = {global_max}")
print(f"global_N   = {global_N}")
print(f"log2(global_N) = {np.log2(global_N)}")
print()

# ------------------------------------------------------------------
# Scaling function
# ------------------------------------------------------------------
def global_scale(opc_array, gmin, gmax, gN):
    if gmax == gmin:
        return np.zeros_like(opc_array)
    return np.log2(gN) * (opc_array - gmin) / (gmax - gmin)

# ------------------------------------------------------------------
# 2. Apply global scaling to each dataset
# ------------------------------------------------------------------
results = []

for file in files:
    df = pd.read_csv(
        file,
        delim_whitespace=True,
        skiprows=1,
        names=["ID", "Cusps", "OPC"]
    )
    
    scaled_values = global_scale(
        df["OPC"].values,
        global_min,
        global_max,
        global_N
    )
    
    mean_scaled = scaled_values.mean()
    
    results.append({
        "file": file.name,
        "sample_size": len(df),
        "mean_scaled_complexity_global": mean_scaled
    })

# Convert to DataFrame for clean display
results_df = pd.DataFrame(results).sort_values("sample_size")

print("Globally Scaled Mean Complexities")
print("------------------------------------------------")
display(results_df)

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({

    # -------- Lines --------
    "lines.linewidth": 2,
    "lines.markersize": 4.5,

    # -------- Axes labels --------
    "axes.labelsize": 14,
    "axes.titlesize": 14,

    # -------- Tick labels --------
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # -------- Legend --------
    "legend.fontsize": 11,

    # -------- Grid --------
    "grid.alpha": 0.25,

})

function A

In [ ]:
def plot_A(ax):
    """
    Tooth — Plot A (self-contained version)
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    # ===================================
    # Configuration
    # ===================================
    base_dir = Path(
        '/Users/sam/Documents/Oxford/Physics/sloppiness/'
        'circadian/mut_project_updates/figures/tooth model/'
        'plot_a/plot_a_files')
    
    cusp_filter = None
    n_bins = 30
    # ===================================
    # Helper functions
    # ===================================

    def get_file_paths(base_dir):
        rand_path = base_dir / "Rand" / "RandOPCandCusp.txt"
        mut0_path = base_dir / "0mut" / "0mut_cusp_and_opc_9000.txt"

        if not rand_path.exists():
            raise FileNotFoundError(rand_path)
        if not mut0_path.exists():
            raise FileNotFoundError(mut0_path)

        return rand_path, mut0_path

    def read_file(path, name):
        df = pd.read_csv(path, sep=r'\s+|\t', engine='python')
        df.columns = df.columns.str.strip()

        for col in ['ID', 'Cusps', 'OPC']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        df = df.dropna()
        return df

    def process_dataset(df, name, cusp_filter):
        if cusp_filter is not None:
            df = df[df['Cusps'] == cusp_filter]

        opc = df['OPC'].values
        opc = opc[opc > 0]
        return opc

    # ===================================
    # Load data
    # ===================================

    rand_path, mut0_path = get_file_paths(base_dir)

    df_rand = read_file(rand_path, "Random")
    df_0_mut = read_file(mut0_path, "Whole_space")

    rand_raw = process_dataset(df_rand, "Random", cusp_filter)
    zero_raw = process_dataset(df_0_mut, "Whole_space", cusp_filter)

    # ===================================
    # Log bins
    # ===================================

    all_data = np.concatenate([rand_raw, zero_raw])
    min_val = all_data.min()
    max_val = all_data.max()

    bins = np.logspace(np.log10(min_val),
                       np.log10(max_val),
                       n_bins)

    rand_counts, bin_edges = np.histogram(rand_raw, bins=bins)
    zero_counts, _ = np.histogram(zero_raw, bins=bins)

    rand_counts = rand_counts / rand_counts.sum()
    zero_counts = zero_counts / zero_counts.sum()

    # ===================================
    # Print distribution means and complexity means
    # ===================================
    g_complexity_mean = np.mean(rand_raw)
    p_complexity_mean = np.mean(zero_raw)
    print(f"G-dist mean complexity: {g_complexity_mean:.6f}")
    print(f"P-dist mean complexity: {p_complexity_mean:.6f}")
    print(f"Number of samples in G-dist: {len(rand_raw)}")
    print(f"Number of samples in P-dist: {len(zero_raw)}")
    print(f"Location of local maxima in G-dist: {bin_edges[np.where((rand_counts[1:-1] > rand_counts[:-2]) & (rand_counts[1:-1] > rand_counts[2:]))[0] + 1]}")

    bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])

    # ===================================
    # Adaptive epsilon floor
    # ===================================

    all_nonzero = np.concatenate([
        rand_counts[rand_counts > 0],
        zero_counts[zero_counts > 0]
    ])

    eps = np.min(all_nonzero) * 0.1
    rand_safe = np.clip(rand_counts, eps, None)
    zero_safe = np.clip(zero_counts, eps, None)

    # ===================================
    # Main Plot
    # ===================================

    ax.plot(bin_centers, rand_counts,
            marker='o',
            color="orange",
            label="G-dist")

    ax.fill_between(bin_centers, rand_counts, 0,
                    color="orange", alpha=0.18)

    ax.plot(bin_centers, zero_counts,
            marker='o',
            color="blue",
            label="P-dist")

    ax.fill_between(bin_centers, zero_counts, 0,
                    color="blue", alpha=0.15)

    ax.set_xscale("log")
    ax.set_xlabel("Complexity (OPCR, log scale)")
    ax.set_ylabel("Relative Frequency")

    ax.legend(frameon=False, loc="upper center")

    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.set_axisbelow(True)

    # ===================================
    # Inset
    # ===================================

    ax_inset = inset_axes(ax, width="22%", height="22%", loc="upper right")

    ax_inset.plot(bin_centers, rand_safe, color="orange")
    ax_inset.plot(bin_centers, zero_safe, color="blue")

    ax_inset.set_xscale("log")
    ax_inset.set_yscale("log")
    ax_inset.set_xlabel("Complexity", fontsize=9)
    ax_inset.set_ylabel("Log Frequency", fontsize=9)
    ax_inset.tick_params(labelsize=8)

function B

In [ ]:
def plot_B(ax):
    """
    Tooth — Plot B
    Entropy and mean complexity vs number of sampled genotypes.
    """

    import numpy as np
    from pathlib import Path

    base_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/tooth model/"
        "plot_b/plot_b_files/plot_c"
    )

    configs = [
        {'N': 10, 'dir': '10'},
        {'N': 100, 'dir': '100'},
        {'N': 1000, 'dir': '103'},
        {'N': 10000, 'dir': '104'},
        {'N': 100000, 'dir': '105'}
    ]

    data = []

    for config in configs:
        N = config['N']
        dir_path = base_dir / config['dir']

        try:
            with open(dir_path / f"{config['dir']}entropy.txt", 'r') as f:
                entropy = float(f.read().strip())

            #this is if i wanted to plot locally scaled mean
            #with open(dir_path / "scaled_mean.txt", 'r') as f:

            #this is if i wanted to plot unscaled scaled mean
            with open(dir_path / "simple_mean.txt", 'r') as f:
                
                scaled_mean = float(f.read().strip())

            data.append((N, entropy, scaled_mean))

        except Exception as e:
            print(f"Warning: Could not read data for N={N}: {e}")

    data.sort(key=lambda x: x[0])

    N_values = [d[0] for d in data]
    entropies = [d[1] for d in data]
    scaled_means = [d[2] for d in data]

    # ===================================
    # Plot
    # ===================================

    ax.semilogx(N_values, entropies, 'o-',
                color='green', label='S')

    ax.semilogx(N_values, scaled_means, 'o-',
                color='orange', label=r'$\langle \tilde{K}_{g} \rangle$')

    ax.set_xlabel('Number of sampled genotypes (N, log scale)')
    ax.set_ylabel(r'Entropy (S) and mean complexity $\langle \tilde{K}_{g} \rangle$')

    ax.set_xlim(5, 200000)

    x_ticks = [10, 100, 1000, 10000, 100000]
    x_tick_labels = ['10¹', '10²', '10³', '10⁴', '10⁵']
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels)
    ax.minorticks_off()
    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    ax.legend(frameon=False, loc='upper left')

def plot_B(ax, results_df):
    """
    Tooth — Plot B
    Entropy and globally scaled mean complexity vs number of sampled genotypes.
    """

    import numpy as np
    from pathlib import Path

    base_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/tooth model/"
        "plot_b/plot_b_files/plot_c"
    )

    configs = [
        {'N': 10, 'dir': '10', 'file': '10OPCandCusp.txt'},
        {'N': 100, 'dir': '100', 'file': '100OPCandCusp.txt'},
        {'N': 1000, 'dir': '103', 'file': '103OPCandCusp.txt'},
        {'N': 10000, 'dir': '104', 'file': '104OPCandCusp.txt'},
        {'N': 100000, 'dir': '105', 'file': '105OPCandCusp.txt'}
    ]

    data = []

    for config in configs:
        N = config['N']
        dir_path = base_dir / config['dir']

        try:
            # -------------------------------
            # Read entropy (already saved)
            # -------------------------------
            with open(dir_path / f"{config['dir']}entropy.txt", 'r') as f:
                entropy = float(f.read().strip())

            # -------------------------------
            # Look up globally scaled mean
            # -------------------------------
            match = results_df[results_df["file"] == config["file"]]

            if len(match) == 0:
                raise ValueError("Global mean not found in results_df")

            scaled_mean = match["mean_scaled_complexity_global"].values[0]

            data.append((N, entropy, scaled_mean))

        except Exception as e:
            print(f"Warning: Could not read data for N={N}: {e}")

    data.sort(key=lambda x: x[0])

    N_values = [d[0] for d in data]
    entropies = [d[1] for d in data]
    scaled_means = [d[2] for d in data]

    # ===================================
    # Plot
    # ===================================

    ax.semilogx(N_values, entropies, 'o-',
                color='green', label='S')

    ax.semilogx(N_values, scaled_means, 'o-',
                color='orange', label=r'$\langle \tilde K_{g} \rangle$')

    ax.set_xlabel('Number of sampled genotypes (N, log scale)')
    ax.set_ylabel(r'Entropy (S) and scaled $\langle \tilde K_{g}\rangle$')

    ax.set_xlim(5, 200000)

    x_ticks = [10, 100, 1000, 10000, 100000]
    x_tick_labels = ['10¹', '10²', '10³', '10⁴', '10⁵']
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels)
    ax.minorticks_off()
    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    ax.legend(frameon=False, loc='upper left')

function C

In [ ]:
def plot_C(ax):
    """
    Tooth — Plot C
    Mean OPC vs Shannon entropy (linear regression).
    """

    import numpy as np
    from pathlib import Path

    data_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/tooth model/"
        "plot_c/plot_c_files/plot_d"
    )

    sizes = [2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000]

    entropy_vals = []
    kc_vals = []
    labels = []

    for n in sizes:
        entropy_file = data_dir / f"{n}_entropy.txt"
        mean_file = data_dir / f"{n}_meanOPC.txt"

        try:
            S = float(entropy_file.read_text().strip())
            K = float(mean_file.read_text().strip())

            entropy_vals.append(S)
            kc_vals.append(K)
            labels.append(str(n))

        except:
            continue

    entropy_vals = np.array(entropy_vals)
    kc_vals = np.array(kc_vals)

    # Linear regression
    m, b = np.polyfit(entropy_vals, kc_vals, 1)
    fit_line = m * entropy_vals + b
    sign = "-" if b < 0 else "+"
    eq_label = rf"$\langle \tilde{{K}}_{{g}} \rangle = {m:.2f}\,S {sign} {abs(b):.2f}$"

    ax.scatter(entropy_vals, kc_vals, color='orange')
    ax.plot(entropy_vals, fit_line, color='orange', label=eq_label)

    offset = 0.05 * (kc_vals.max() - kc_vals.min())

    for xi, yi, lbl in zip(entropy_vals, kc_vals, labels):
        ax.text(xi, yi + offset, lbl, fontsize=11, ha='center', va='top')

    ax.set_xlabel("Shannon Entropy (S)")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")

    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.margins(x=0.05, y=0.05)
    ax.legend(frameon=False)

function D

In [ ]:
def plot_D(ax,
           base_dir=None,
           gen0_file=None,
           gen0_contents=None,
           gen0_override=None,
           verbose=True):
    """
    Robust plotting of mean OPC vs mutation number grouped by quintile.

    Use:
      plot_D(ax)  # uses defaults
      plot_D(ax, gen0_file="/full/path/rand_quintile_means.txt")
      plot_D(ax, gen0_contents="Q1 6.27 Q2 9.40 Q3 10.55 Q4 12.78 Q5 117.606386.")
      plot_D(ax, gen0_override={"Q1":6.27,...})
    Returns parsed gen0_means dict.
    """
    import numpy as np
    from pathlib import Path
    import matplotlib.ticker as mticker
    import re
    import os

    # ---------- defaults ----------
    default_base_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
        "mut_project_updates/figures/tooth model/plot_d/plot_d_files"
    )
    default_gen0_file = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
        "mut_project_updates/figures/tooth/rand_quintile_means.txt"
    )

    base_dir = Path(base_dir) if base_dir is not None else default_base_dir
    gen0_file = Path(gen0_file) if gen0_file is not None else default_gen0_file

    # If user supplied override dict, use it immediately
    if gen0_override is not None:
        gen0_means = {}
        for i in range(1, 6):
            k = f"Q{i}"
            v = gen0_override.get(k, np.nan)
            try:
                gen0_means[k] = float(v)
            except Exception:
                gen0_means[k] = np.nan
        if verbose:
            print("Used gen0_override dict:", gen0_means)
        _do_plot(ax, base_dir, gen0_means, verbose)
        return gen0_means

    # Regexes
    # Pair regex: finds all occurrences like "Q1 6.27", "Q2:9.4", "q5=117.606386."
    pair_re = re.compile(
        r'(Q[1-5])\s*[:=]?\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)(?=[^\d\.eE+-]|$)',
        re.IGNORECASE,
    )
    # generic float regex for fallbacks
    float_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
    qkey_re = re.compile(r"(Q[1-5])", re.IGNORECASE)

    def parse_gen0_text(text):
        # 1) try pair-based extraction across entire text
        d = {f"Q{i}": np.nan for i in range(1, 6)}
        found_any = False
        for m in pair_re.finditer(text):
            key = m.group(1).upper()
            num_s = m.group(2)
            try:
                d[key] = float(num_s)
            except Exception:
                # last-ditch: remove trailing punctuation and try again
                cleaned = re.sub(r"[^\d\.\-+eE]", "", num_s)
                try:
                    d[key] = float(cleaned)
                except Exception:
                    d[key] = np.nan
            found_any = True

        if found_any:
            return d

        # 2) fallback: token scan (robust)
        # build token list, then walk tokens looking for Qn followed by a numeric token
        tokens = re.split(r"[,\s;]+", text.strip())
        for i, t in enumerate(tokens):
            k_m = qkey_re.fullmatch(t)
            if k_m:
                key = k_m.group(1).upper()
                # look ahead for a numeric token
                val = np.nan
                if i + 1 < len(tokens):
                    nxt = tokens[i + 1]
                    f_m = float_re.search(nxt)
                    if f_m:
                        try:
                            val = float(f_m.group())
                        except Exception:
                            val = np.nan
                d[key] = val
        return d

    # If gen0_contents provided, parse immediately
    if gen0_contents is not None:
        gen0_means = parse_gen0_text(gen0_contents)
        if verbose:
            print("Parsed gen0_means from provided contents:", gen0_means)
        _do_plot(ax, base_dir, gen0_means, verbose)
        return gen0_means

    # Locate file (try provided path, then fallback locations)
    tried = []
    found_path = None
    if gen0_file.exists():
        found_path = gen0_file.resolve()
        tried.append(str(gen0_file))
    else:
        tried.append(str(gen0_file))
        candidates = [
            base_dir / "rand_quintile_means.txt",
            base_dir.parent / "rand_quintile_means.txt",
            default_gen0_file,
        ]
        for c in candidates:
            tried.append(str(c))
            if c.exists():
                found_path = c.resolve()
                break

        # shallow recursive search if still not found
        if found_path is None and base_dir.exists():
            max_depth = 4
            base_parts = base_dir.resolve().parts
            for root, dirs, files in os.walk(base_dir):
                # depth pruning
                if len(Path(root).resolve().parts) - len(base_parts) > max_depth:
                    dirs[:] = []
                    continue
                for fname in files:
                    fpath = Path(root) / fname
                    tried.append(str(fpath))
                    if "rand_quintile" in fname.lower():
                        found_path = fpath.resolve()
                        break
                    if fpath.suffix.lower() == ".txt":
                        try:
                            with fpath.open("r", encoding="utf-8", errors="ignore") as fh:
                                head = "".join([next(fh) for _ in range(10)])
                            if re.search(r"Q[1-5]\b", head, re.IGNORECASE):
                                found_path = fpath.resolve()
                                break
                        except StopIteration:
                            pass
                        except Exception:
                            pass
                if found_path is not None:
                    break

    if found_path is None:
        if verbose:
            print("Warning: rand_quintile_means.txt not found (sample locations tried):")
            for p in tried[:12]:
                print("  -", p)
            print("You can pass gen0_file explicitly or supply gen0_contents or gen0_override.")
        gen0_means = {f"Q{i}": np.nan for i in range(1, 6)}
        _do_plot(ax, base_dir, gen0_means, verbose)
        return gen0_means

    # read file and parse all pairs
    try:
        raw = found_path.read_text(encoding="utf-8", errors="ignore")
        gen0_means = parse_gen0_text(raw)
        if verbose:
            print("Using rand_quintile_means.txt at:", str(found_path))
            print("Parsed gen0_means:", gen0_means)
    except Exception as e:
        if verbose:
            print(f"Warning: failed to read {found_path}: {e}")
        gen0_means = {f"Q{i}": np.nan for i in range(1, 6)}

    _do_plot(ax, base_dir, gen0_means, verbose)
    return gen0_means


def _do_plot(ax, base_dir, gen0_means, verbose):
    import numpy as np
    from pathlib import Path
    import matplotlib.ticker as mticker
    import re

    base_dir = Path(base_dir)

    Qs = [f"Q{i}" for i in range(1, 6)]
    muts = [f"mut{i}" for i in range(1, 26)]
    n_mut_files = len(muts)
    x_positions = np.arange(0, n_mut_files + 1)

    float_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

    def read_numeric_file(path):
        try:
            text = Path(path).read_text(encoding="utf-8", errors="ignore").strip()
            if not text:
                return None
            tokens = re.split(r"[,\s;]+", text)
            nums = []
            for t in tokens:
                try:
                    nums.append(float(t))
                    continue
                except Exception:
                    pass
                m = float_re.search(t)
                if m:
                    try:
                        nums.append(float(m.group()))
                    except Exception:
                        continue
            if not nums:
                return None
            return float(np.mean(nums))
        except FileNotFoundError:
            return None
        except Exception:
            return None

    all_values = []
    for q in Qs:
        vals = []
        for m in muts:
            path = base_dir / q / m / f"{q}_{m}_meanOPC.txt"
            v = read_numeric_file(path)
            vals.append(v if v is not None else np.nan)
        all_values.append(vals)

    all_values = np.array(all_values, dtype=float)

    palette = [
        "#E8B5D6",
        "#CC79A7",
        "#A63478",
        "#7A1F5C",
        "#4A0D3A",
    ]

    last_series = None
    for i, q in enumerate(Qs):
        y25 = all_values[i]
        gen0_val = gen0_means.get(q, np.nan)
        values = np.insert(y25, 0, gen0_val)
        last_series = values.copy()
        ax.plot(x_positions, values, marker="o", color=palette[i], label=f"G{i+1}", linestyle="-")

    ax.set_xlabel("Number of mutations")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")
    ax.xaxis.set_major_locator(mticker.MultipleLocator(4))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    # determine spacing dynamically
    if len(x_positions) > 1:
        spacing = x_positions[1] - x_positions[0]
    else:
        spacing = 1  # fallback

    ax.set_xlim(
        min(x_positions) - 0.5 * spacing,
        max(x_positions) + 0.5 * spacing
    )
    ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.set_axisbelow(True)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[::-1], labels[::-1], title="Groups", loc="upper right", frameon=False)
    ax.margins(x=0.02, y=0.05)

    if verbose:
        print("First x value:", int(x_positions[0]))
        if last_series is not None:
            print("First y value (last plotted series):", last_series[0])
        else:
            print("No series plotted.")

create master figure

In [ ]:
import matplotlib.pyplot as plt

# ===================================
# Create master 2x2 figure
# ===================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes = axes.flatten()

# Call your plotting functions
plot_A(axes[0])
plot_B(axes[1])
#plot_B(axes[1], results_df)   # <-- pass globally scaled results here
plot_C(axes[2])
plot_D(axes[3])

# ===================================
# Panel labels underneath (standardized)
# ===================================
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for ax, label in zip(axes, panel_labels):
    ax.text(
        0.5,
        -0.14,
        label,
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=11
    )

plt.tight_layout()
plt.subplots_adjust(hspace=0.28, wspace=0.25)

####### notes for C#plt.show()

#### notes for A
- [x] x - axis: Complexity ( type )
- [x] capitol N 
- [x] 2-mut
- [x] 1-mut
- [x] 0-mut
- [x] y axis label: Relative frequency OR log frequency 
- [x] G-dist

#### notes for B
Legend: S, and Kg
y - axis: Entropy (S) and mean complexity (Kg)
Number of sampled genotypes (N)
All markers circular across all plots (B and D)


#### notes for C
x-axis (Shannon Entropy (S))
Mean Complexity (Kg)
Circle markers 
red line: linear fit

#### notes for D:

y axis mean complexity (Kg)
keep grid
all circular markers
x-axis : number 
Groups
G5 descending to G1
0.5 white margin on either side of all data points 
